<a href="https://colab.research.google.com/github/hadiah115-tech/Urdu-OCR_Project/blob/main/SI26_Week_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install gradio

In [5]:
!pip install -q gradio==5.38.2 transformers torch pillow sentencepiece

In [6]:
import gradio
print(gradio.__version__)

5.38.2


In [7]:
!pip show gradio

Name: gradio
Version: 5.38.2
Summary: Python library for easily interacting with trained machine learning models
Home-page: https://github.com/gradio-app/gradio
Author: 
Author-email: Abubakar Abid <gradio-team@huggingface.co>, Ali Abid <gradio-team@huggingface.co>, Ali Abdalla <gradio-team@huggingface.co>, Dawood Khan <gradio-team@huggingface.co>, Ahsen Khaliq <gradio-team@huggingface.co>, Pete Allen <gradio-team@huggingface.co>, Ömer Faruk Özdemir <gradio-team@huggingface.co>, Freddy A Boulton <gradio-team@huggingface.co>, Hannah Blair <gradio-team@huggingface.co>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: aiofiles, anyio, brotli, fastapi, ffmpy, gradio-client, groovy, httpx, huggingface-hub, jinja2, markupsafe, numpy, orjson, packaging, pandas, pillow, pydantic, pydub, python-multipart, pyyaml, ruff, safehttpx, semantic-version, starlette, tomlkit, typer, typing-extensions, uvicorn
Required-by: 


In [8]:
import gradio as gr
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image

print("Gradio:", gr.__version__)
print("Torch:", torch.__version__)

Gradio: 5.38.2
Torch: 2.11.0+cpu


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
model_path = "/content/drive/MyDrive/SI26-urdu-ocr-model"

processor = TrOCRProcessor.from_pretrained(model_path)
model = VisionEncoderDecoderModel.from_pretrained(model_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print("✅ Model Loaded Successfully!")

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

✅ Model Loaded Successfully!


In [11]:
def extract_urdu_text(image):
    if image is None:
        return "Please upload an image."

    pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)

    with torch.no_grad():
        generated_ids = model.generate(pixel_values)

    text = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    if text.strip() == "":
        return "Could not extract text."

    return text

In [20]:
demo = gr.Interface(
    fn=extract_urdu_text,
    inputs=gr.Image(type="pil", label="Upload Urdu Image"),
    outputs=gr.Textbox(label="Extracted Urdu Text"),
    title="Urdu OCR - Code Saviours SI-26",
    description="Upload an image containing Urdu text and extract the text."
)

In [ ]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://09d4bddb6afe4600e7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1625: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1625: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1625: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


In [14]:
print(processor)

TrOCRProcessor:
- image_processor: ViTImageProcessor {
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.5,
    0.5,
    0.5
  ],
  "image_processor_type": "ViTImageProcessor",
  "image_std": [
    0.5,
    0.5,
    0.5
  ],
  "resample": 2,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "height": 384,
    "width": 384
  }
}

- tokenizer: TokenizersBackend(name_or_path='/content/drive/MyDrive/SI26-urdu-ocr-model', vocab_size=50265, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False,

In [15]:
import os

model_path = "/content/drive/MyDrive/SI26-urdu-ocr-model"

print(os.listdir(model_path))

['config.json', 'generation_config.json', 'special_tokens_map.json', 'tokenizer_config.json', 'preprocessor_config.json', 'vocab.json', 'merges.txt', 'model.safetensors', 'tokenizer.json']


In [16]:
# 1. Login to Hugging Face
from huggingface_hub import notebook_login
notebook_login()

In [17]:
# 2. Push your model + processor to the Hub
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

model_path = '/content/drive/MyDrive/SI26-urdu-ocr-model'
processor = TrOCRProcessor.from_pretrained(model_path)
model = VisionEncoderDecoderModel.from_pretrained(model_path)

repo_name = "urdu-ocr-si26-hadia"  # change to your name, keep it unique

processor.push_to_hub(repo_name)
model.push_to_hub(repo_name)

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

No files have been modified since last commit. Skipping to prevent empty commit.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...9zbc3nr/model.safetensors:   1%|          | 7.95MB / 1.34GB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/hadia-tech/urdu-ocr-si26-hadia/commit/cb41cc89d2cc67697fd7e515181fcf3647ab4ac0', commit_message='Upload model', commit_description='', oid='cb41cc89d2cc67697fd7e515181fcf3647ab4ac0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/hadia-tech/urdu-ocr-si26-hadia', endpoint='https://huggingface.co', repo_type='model', repo_id='hadia-tech/urdu-ocr-si26-hadia'), pr_revision=None, pr_num=None)